In [ ]:
# 【环境限制提示，非代码 bug】panel 是用来搭建本 notebook 最后交互式聊天 Dashboard 的 GUI 库，
# 当前 venv 未安装 panel（以及它依赖的 param），import 会报 ModuleNotFoundError。
# 这与 LangChain 版本升级无关，纯粹是这个可选的 GUI 依赖没有装；如果只关心 RAG 本身的问答逻辑
# （前半部分 qa_chain / ConversationalRetrievalChain），可以先跳过安装，等跑到最后 Dashboard 部分再
# `pip install panel param` 即可。
import os
import openai
import sys
sys.path.append('../..')

import panel as pn  # GUI
pn.extension()

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

openai.api_key  = os.environ['OPENAI_API_KEY']

**RAG 流程小结**：本 notebook 在 05 节问答的基础上，加入**对话记忆（Memory）**，把无状态的一问一答升级成
真正的多轮对话机器人——`ConversationalRetrievalChain` 会先结合历史对话把当前问题"改写"成一个独立、完整的问题
（比如把"why are those prerequisites needed?"改写成"为什么 XX 课程需要这些先修要求？"），
再拿改写后的问题去做检索、生成回答，从而解决 05 节末尾发现的"检索器不理解上下文"的问题。
最后还会用 `panel` 搭一个可交互的网页聊天界面（Dashboard）。

In [ ]:
# 同 05 节：根据当前日期自动选用合适的 OpenAI 模型名，避免用到已下线的旧版本号
import datetime
current_date = datetime.datetime.now().date()
if current_date < datetime.date(2023, 9, 2):
    llm_name = "gpt-3.5-turbo-0301"
else:
    llm_name = "gpt-3.5-turbo"
print(llm_name)

In [ ]:
# 需要用 LangSmith 追踪调用链路细节的话，取消注释并填入真实 API Key
#import os
#os.environ["LANGCHAIN_TRACING_V2"] = "true"
#os.environ["LANGCHAIN_ENDPOINT"] = "https://api.langchain.plus"
#os.environ["LANGCHAIN_API_KEY"] = "..."

In [ ]:
# 【真实 Bug 修复】打开前面几节已经持久化好的 Chroma 向量库。
#
# 原代码 `from langchain_core.vectorstores import Chroma` 是错的：langchain_core.vectorstores 里
# 只有 VectorStore 抽象基类，没有 Chroma 这个具体实现，会 ImportError（和 05 节同样的错误）。
# `from langchain_community.embeddings.openai import OpenAIEmbeddings` 虽然还能 import（带 deprecation warning），
# 但为了和其它 notebook 保持一致，统一改用官方推荐的独立包 langchain_openai。
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
persist_directory = 'docs/chroma/'
embedding = OpenAIEmbeddings()
vectordb = Chroma(persist_directory=persist_directory, embedding_function=embedding)

In [ ]:
# 验证向量库能正常检索
question = "What are major topics for this class?"
docs = vectordb.similarity_search(question,k=3)
len(docs)

In [ ]:
# 【真实 Bug 修复】ChatOpenAI 正确路径同样是 langchain_openai（旧版 langchain_community.chat_models 已不存在）。
#
# 另外 `.predict("Hello world!")` 也是真实 bug：predict() 是旧版 BaseChatModel 上的便捷方法，
# 在当前 langchain-core 里 BaseChatModel 已经完全移除了 predict/predict_messages 这些旧接口，
# 只保留统一的 Runnable 接口 .invoke(...)。用 .predict() 会直接 AttributeError。
# 改用 .invoke(...) 后，返回值不再是纯字符串，而是一个 AIMessage 对象，取文本内容要用 .content。
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model_name=llm_name, temperature=0)
llm.invoke("Hello world!").content

In [ ]:
# 复用 05 节的自定义 prompt 和不带记忆的 RetrievalQA，作为下面"带记忆版本"的对比基线
# Build prompt
from langchain_core.prompts import PromptTemplate
template = """Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer. Use three sentences maximum. Keep the answer as concise as possible. Always say "thanks for asking!" at the end of the answer.
{context}
Question: {question}
Helpful Answer:"""
QA_CHAIN_PROMPT = PromptTemplate(input_variables=["context", "question"],template=template,)

# 【真实 Bug 修复】RetrievalQA 正确路径是 langchain_classic.chains
# （原代码 `from langchain_community.chains import RetrievalQA` 在当前 langchain_community 0.4.2 下会 ImportError）
# Run chain
from langchain_classic.chains import RetrievalQA
question = "Is probability a class topic?"
qa_chain = RetrievalQA.from_chain_type(llm,
                                       retriever=vectordb.as_retriever(),
                                       return_source_documents=True,
                                       chain_type_kwargs={"prompt": QA_CHAIN_PROMPT})


result = qa_chain({"query": question})
result["result"]

In [ ]:
# ConversationBufferMemory：最简单的对话记忆实现，把历史的每一轮问答原样存进一个列表里，
# 之后每次调用链时都会把这些历史消息一起传给 LLM，让它"记得"之前聊过什么。
# memory_key="chat_history" 要和链（这里是下面的 ConversationalRetrievalChain）内部读取历史记录时用的变量名对应上，
# return_messages=True 表示存的是消息对象列表（更适合 Chat 模型），而不是拼接好的一整段文本。
#
# 路径说明：langchain_classic.memory 是当前 langchain 1.4.0 下 Memory 相关组件的正确位置
# （旧版课程写法是 `from langchain.memory import ConversationBufferMemory`，这个路径在新版下已不存在）。
from langchain_classic.memory import ConversationBufferMemory
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

In [ ]:
# 【真实 Bug 修复】ConversationalRetrievalChain：带记忆的检索问答链，核心区别于 RetrievalQA 的地方是
# 它会先结合 memory 里的历史对话，把当前这一轮的问题"改写"成一个独立、完整的问题，再去做检索，
# 这样即使问题里有代词/省略（比如 "why are those prerequisites needed?"）也能正确检索到相关内容。
#
# 原代码 `from langchain_community.chains import ConversationalRetrievalChain` 会 ImportError
# （这个类不在 langchain_community.chains 里），正确路径同 RetrievalQA，都在 langchain_classic.chains。
from langchain_classic.chains import ConversationalRetrievalChain
retriever=vectordb.as_retriever()
qa = ConversationalRetrievalChain.from_llm(
    llm,
    retriever=retriever,
    memory=memory
)

In [ ]:
# 第一轮提问：因为传了 memory，这一轮的问答会自动被记录下来
question = "Is probability a class topic?"
result = qa({"question": question})

In [ ]:
result['answer']

In [ ]:
# 第二轮提问：这句里的 "those prerequisites" 指代上一轮的内容。
# 因为有 memory，ConversationalRetrievalChain 会先把它改写成类似"这门课为什么需要概率这样的先修知识？"的独立问题，
# 再去检索，效果应该比 05 节里不带记忆的 RetrievalQA 好很多
question = "why are those prerequesites needed?"
result = qa({"question": question})

In [ ]:
result['answer']

In [ ]:
# 汇总一下下面要用到的所有组件的正确导入路径（和前面每个 cell 里解释的一致）：
# - OpenAIEmbeddings / ChatOpenAI：langchain_openai（旧版 langchain_community.* 路径已不可用或已弃用）
# - RetrievalQA / ConversationalRetrievalChain：langchain_classic.chains（旧版 langchain_community.chains 里没有）
# - ConversationBufferMemory：langchain_classic.memory
#
# 【真实 Bug 修复 + 环境限制提示】DocArrayInMemorySearch 是一个纯内存向量库（不需要额外部署数据库），
# 依赖 docarray 这个第三方包。当前 venv 没有安装 docarray，实际调用 .from_documents() 时会报错。
# 这是缺依赖问题，不是 langchain 版本升级的锅；如果只是练习代码结构，import 语句本身不会报错，
# 但真正跑 load_db() 时需要先 `pip install docarray`。
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_community.vectorstores import DocArrayInMemorySearch
from langchain_community.document_loaders import TextLoader
from langchain_classic.chains import RetrievalQA, ConversationalRetrievalChain
from langchain_classic.memory import ConversationBufferMemory
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import PyPDFLoader

In [ ]:
# load_db：把"加载 PDF -> 切分 -> 建向量库 -> 建带记忆的问答链"整个 RAG 流程封装成一个函数，
# 方便后面 Dashboard 里用户上传新 PDF 时可以重新构建一整套问答能力
def load_db(file, chain_type, k):
    # load documents
    loader = PyPDFLoader(file)
    documents = loader.load()
    # split documents
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
    docs = text_splitter.split_documents(documents)
    # define embedding
    embeddings = OpenAIEmbeddings()
    # create vector database from data
    db = DocArrayInMemorySearch.from_documents(docs, embeddings)
    # define retriever
    retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": k})
    # create a chatbot chain. Memory is managed externally.
    # 注意：这里没有传 memory 参数，而是通过 return_generated_question=True 让调用方（下面的 convchain）
    # 自己维护 chat_history 并手动传入，这样可以在 Dashboard 里灵活地清空/展示对话历史
    qa = ConversationalRetrievalChain.from_llm(
        llm=ChatOpenAI(model_name=llm_name, temperature=0),
        chain_type=chain_type,
        retriever=retriever,
        return_source_documents=True,
        return_generated_question=True,
    )
    return qa


In [ ]:
# cbfs（ChatBot File System 的缩写）：用 param.Parameterized 定义一个响应式的状态类，
# 每个 param.XXX(...) 声明的属性一旦变化，绑定它的 UI 组件（下面 get_lquest/get_sources/get_chats）会自动重新渲染，
# 这是 panel 这套响应式 GUI 框架的核心用法。
import panel as pn
import param

class cbfs(param.Parameterized):
    chat_history = param.List([])
    answer = param.String("")
    db_query  = param.String("")
    db_response = param.List([])

    def __init__(self,  **params):
        super(cbfs, self).__init__( **params)
        self.panels = []
        # 默认预加载课程自带的第一讲 PDF，用户也可以在 Dashboard 里上传自己的 PDF 替换掉它
        self.loaded_file = "docs/cs229_lectures/MachineLearning-Lecture01.pdf"
        self.qa = load_db(self.loaded_file,"stuff", 4)

    def call_load_db(self, count):
        if count == 0 or file_input.value is None:  # init or no file specified :
            return pn.pane.Markdown(f"Loaded File: {self.loaded_file}")
        else:
            file_input.save("temp.pdf")  # local copy
            self.loaded_file = file_input.filename
            button_load.button_style="outline"
            self.qa = load_db("temp.pdf", "stuff", 4)
            button_load.button_style="solid"
        self.clr_history()
        return pn.pane.Markdown(f"Loaded File: {self.loaded_file}")

    def convchain(self, query):
        if not query:
            return pn.WidgetBox(pn.Row('User:', pn.pane.Markdown("", width=600)), scroll=True)
        # 手动把 self.chat_history 传给 qa，因为 load_db() 里构造 ConversationalRetrievalChain 时没有传 memory，
        # 对话历史完全由这个 cbfs 类自己维护（方便配合"Clear History"按钮随时清空）
        result = self.qa({"question": query, "chat_history": self.chat_history})
        self.chat_history.extend([(query, result["answer"])])
        self.db_query = result["generated_question"]
        self.db_response = result["source_documents"]
        self.answer = result['answer']
        self.panels.extend([
            pn.Row('User:', pn.pane.Markdown(query, width=600)),
            pn.Row('ChatBot:', pn.pane.Markdown(self.answer, width=600, style={'background-color': '#F6F6F6'}))
        ])
        inp.value = ''  #clears loading indicator when cleared
        return pn.WidgetBox(*self.panels,scroll=True)

    # 【真实 Bug 修复】原代码是 @param.depends('db_query ', ) —— 注意字符串里 'db_query ' 多了一个尾随空格，
    # 和上面真正声明的参数名 db_query（没有空格）对不上，会导致这个响应式依赖绑定失效
    # （param 库无法把它识别成 db_query 这个参数，这个函数就不会在 db_query 更新时自动重新渲染）。
    # 这里去掉多余的空格修复。
    @param.depends('db_query', )
    def get_lquest(self):
        if not self.db_query :
            return pn.Column(
                pn.Row(pn.pane.Markdown(f"Last question to DB:", styles={'background-color': '#F6F6F6'})),
                pn.Row(pn.pane.Str("no DB accesses so far"))
            )
        return pn.Column(
            pn.Row(pn.pane.Markdown(f"DB query:", styles={'background-color': '#F6F6F6'})),
            pn.pane.Str(self.db_query )
        )

    @param.depends('db_response', )
    def get_sources(self):
        if not self.db_response:
            return
        rlist=[pn.Row(pn.pane.Markdown(f"Result of DB lookup:", styles={'background-color': '#F6F6F6'}))]
        for doc in self.db_response:
            rlist.append(pn.Row(pn.pane.Str(doc)))
        return pn.WidgetBox(*rlist, width=600, scroll=True)

    @param.depends('convchain', 'clr_history')
    def get_chats(self):
        if not self.chat_history:
            return pn.WidgetBox(pn.Row(pn.pane.Str("No History Yet")), width=600, scroll=True)
        rlist=[pn.Row(pn.pane.Markdown(f"Current Chat History variable", styles={'background-color': '#F6F6F6'}))]
        for exchange in self.chat_history:
            rlist.append(pn.Row(pn.pane.Str(exchange)))
        return pn.WidgetBox(*rlist, width=600, scroll=True)

    def clr_history(self,count=0):
        self.chat_history = []
        return


In [ ]:
# 组装最终的交互式 Dashboard：
# - tab1"Conversation"：聊天输入框 + 对话气泡展示
# - tab2"Database"：展示最近一次实际发给向量库的检索问题（db_query）和检索到的原始文档（db_response）
# - tab3"Chat History"：展示完整的 chat_history 变量内容，方便调试
# - tab4"Configure"：允许用户上传自己的 PDF 替换默认课程 PDF，以及清空历史
# 【环境限制提示】这整个 cell 依赖 panel（pn.bind、pn.Tabs 等），当前 venv 未安装 panel/param，会 ModuleNotFoundError，
# 参见本 notebook第一个 cell 的说明；这里保持课程原有结构，不做改写。
cb = cbfs()

file_input = pn.widgets.FileInput(accept='.pdf')
button_load = pn.widgets.Button(name="Load DB", button_type='primary')
button_clearhistory = pn.widgets.Button(name="Clear History", button_type='warning')
button_clearhistory.on_click(cb.clr_history)
inp = pn.widgets.TextInput( placeholder='Enter text here…')

bound_button_load = pn.bind(cb.call_load_db, button_load.param.clicks)
conversation = pn.bind(cb.convchain, inp)

jpg_pane = pn.pane.Image( './img/convchain.jpg')

tab1 = pn.Column(
    pn.Row(inp),
    pn.layout.Divider(),
    pn.panel(conversation,  loading_indicator=True, height=300),
    pn.layout.Divider(),
)
tab2= pn.Column(
    pn.panel(cb.get_lquest),
    pn.layout.Divider(),
    pn.panel(cb.get_sources ),
)
tab3= pn.Column(
    pn.panel(cb.get_chats),
    pn.layout.Divider(),
)
tab4=pn.Column(
    pn.Row( file_input, button_load, bound_button_load),
    pn.Row( button_clearhistory, pn.pane.Markdown("Clears chat history. Can use to start a new topic" )),
    pn.layout.Divider(),
    pn.Row(jpg_pane.clone(width=400))
)
dashboard = pn.Column(
    pn.Row(pn.pane.Markdown('# ChatWithYourData_Bot')),
    pn.Tabs(('Conversation', tab1), ('Database', tab2), ('Chat History', tab3),('Configure', tab4))
)
dashboard